# Module 3.1: Memory Patterns for Agentic AI

<span class="badge blue">Agentic AI</span> <span class="badge amber">~15 min</span> <span class="badge mint">Hands-on</span>

## Learning Objectives
By completing this notebook, you will:
1. Implement short-term memory for maintaining conversation context
2. Build long-term memory systems for persistent agent knowledge
3. Create episodic memory for storing and retrieving specific experiences
4. Design memory management strategies (retention, summarization, pruning)
5. Measure memory impact on agent performance and token usage

## Prerequisites
- Module 1: LLM Fundamentals (system prompts, reasoning patterns)
- Understanding of data structures (lists, dictionaries, queues)
- Basic knowledge of embeddings and similarity

## Estimated Time: 55 minutes

---

## Why Memory Matters for Agents

LLMs are stateless - they don't remember previous interactions without explicit memory systems. Agents need memory for:

- **Conversation continuity**: Referring back to earlier discussion points
- **Learning from experience**: Improving based on past successes/failures
- **Personalization**: Adapting to user preferences over time
- **Complex tasks**: Maintaining state across multi-step workflows

**Real-world application:** Production agents use memory systems to provide coherent multi-turn conversations, remember user preferences, and learn from interactions without retraining models.

**Key insight:** Effective memory management is about knowing what to remember, what to forget, and how to retrieve relevant information efficiently.

## Setup & Imports

In [ ]:
# Setup: Course styling and plot theme
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname("__file__"), "../../../../.."))

from resources.notebook_style import apply_course_theme
from resources.graphics.plot_theme import apply_plot_theme

apply_course_theme()
apply_plot_theme()
print("Course theme applied.")

In [ ]:
# Purpose: Import libraries for memory pattern implementations
# Key Concept: Different memory types require different data structures

import os
import json
from typing import Dict, List, Any, Optional, Tuple
from dataclasses import dataclass, field
from datetime import datetime, timedelta
from collections import deque
from openai import OpenAI
import numpy as np

# Initialize OpenAI client
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

MODEL = "gpt-4o-mini"
EMBEDDING_MODEL = "text-embedding-3-small"

def call_llm(messages: List[Dict[str, str]], temperature: float = 0.0) -> str:
    """Call OpenAI with message history."""
    response = client.chat.completions.create(
        model=MODEL,
        temperature=temperature,
        messages=messages
    )
    return response.choices[0].message.content

def get_embedding(text: str) -> List[float]:
    """Get embedding vector for text."""
    response = client.embeddings.create(
        model=EMBEDDING_MODEL,
        input=text
    )
    return response.data[0].embedding

def cosine_similarity(a: List[float], b: List[float]) -> float:
    """Calculate cosine similarity between two vectors."""
    a_arr = np.array(a)
    b_arr = np.array(b)
    return np.dot(a_arr, b_arr) / (np.linalg.norm(a_arr) * np.linalg.norm(b_arr))

print("✓ Environment configured for memory systems")

## Part 1: Short-Term Memory - Conversation Context

Short-term memory maintains recent conversation history. Key considerations:

- **Window size**: How many recent messages to keep
- **Token limits**: Stay within model context window
- **Summarization**: Compress old messages when window is full

In [ ]:
# Purpose: Implement short-term memory with windowing
# Key Concept: Recent context is most important for conversation coherence

@dataclass
class Message:
    """A single conversation message."""
    role: str  # "user" or "assistant"
    content: str
    timestamp: datetime = field(default_factory=datetime.now)
    metadata: Dict[str, Any] = field(default_factory=dict)


class ShortTermMemory:
    """
    Short-term conversation memory with windowing.
    """
    
    def __init__(
        self,
        max_messages: int = 10,
        system_prompt: Optional[str] = None
    ):
        """
        Initialize short-term memory.
        
        Parameters
        ----------
        max_messages : int
            Maximum number of messages to retain
        system_prompt : Optional[str]
            System prompt (always included, not counted in window)
        """
        self.max_messages = max_messages
        self.system_prompt = system_prompt
        self.messages = deque(maxlen=max_messages)
    
    def add_message(self, role: str, content: str, metadata: Optional[Dict] = None) -> None:
        """
        Add a message to memory.
        
        Parameters
        ----------
        role : str
            Message role ("user" or "assistant")
        content : str
            Message content
        metadata : Optional[Dict]
            Additional metadata
        """
        message = Message(
            role=role,
            content=content,
            metadata=metadata or {}
        )
        self.messages.append(message)
    
    def get_context(self, include_system: bool = True) -> List[Dict[str, str]]:
        """
        Get conversation context in OpenAI format.
        
        Parameters
        ----------
        include_system : bool
            Whether to include system prompt
        
        Returns
        -------
        List[Dict[str, str]]
            Messages in OpenAI format
        """
        context = []
        
        if include_system and self.system_prompt:
            context.append({"role": "system", "content": self.system_prompt})
        
        for msg in self.messages:
            context.append({"role": msg.role, "content": msg.content})
        
        return context
    
    def clear(self) -> None:
        """Clear all messages."""
        self.messages.clear()
    
    def get_recent(self, n: int = 5) -> List[Message]:
        """Get n most recent messages."""
        return list(self.messages)[-n:]


# Test short-term memory
memory = ShortTermMemory(
    max_messages=6,
    system_prompt="You are a helpful coding assistant."
)

# Simulate a conversation
conversation = [
    ("user", "How do I read a file in Python?"),
    ("assistant", "Use open() with a file path: open('file.txt', 'r').read()"),
    ("user", "What if the file doesn't exist?"),
    ("assistant", "Wrap it in try-except to catch FileNotFoundError"),
    ("user", "Can you show me an example?"),
]

for role, content in conversation:
    memory.add_message(role, content)

print("Conversation Context:")
print("=" * 60)
context = memory.get_context()
for msg in context:
    print(f"{msg['role'].upper()}: {msg['content']}")

print(f"\n{len(memory.messages)} messages in memory (max: {memory.max_messages})")

In [ ]:
# Purpose: Test short-term memory in conversation
# Key Concept: Agent uses recent context to maintain coherence

# Continue conversation with memory
user_input = "Yes, please show the example with error handling"
memory.add_message("user", user_input)

# Get response using memory context
response = call_llm(memory.get_context())
memory.add_message("assistant", response)

print("\nAgent Response:")
print("=" * 60)
print(response)

# Test memory window
print("\n\nAdding more messages to test windowing:")
for i in range(5):
    memory.add_message("user", f"Message {i}")
    memory.add_message("assistant", f"Response {i}")

print(f"Messages in memory: {len(memory.messages)} (max: {memory.max_messages})")
print("Recent messages:")
for msg in memory.get_recent(3):
    print(f"  {msg.role}: {msg.content}")

## Part 2: Long-Term Memory - Persistent Knowledge

Long-term memory stores information beyond the conversation window:

- **User preferences**: Settings, communication style, domain knowledge
- **Facts and entities**: Key information extracted from conversations
- **Learned patterns**: Successful strategies and approaches

In [ ]:
# Purpose: Implement long-term memory with structured storage
# Key Concept: Organize memory by type for efficient retrieval

@dataclass
class MemoryEntry:
    """A single long-term memory entry."""
    key: str
    value: Any
    category: str
    created_at: datetime = field(default_factory=datetime.now)
    updated_at: datetime = field(default_factory=datetime.now)
    access_count: int = 0
    importance: float = 1.0  # 0-1 scale


class LongTermMemory:
    """
    Long-term memory for persistent agent knowledge.
    """
    
    def __init__(self):
        """Initialize long-term memory."""
        self.memories: Dict[str, MemoryEntry] = {}
        self.categories: Dict[str, List[str]] = {}
    
    def store(
        self,
        key: str,
        value: Any,
        category: str = "general",
        importance: float = 1.0
    ) -> None:
        """
        Store information in long-term memory.
        
        Parameters
        ----------
        key : str
            Unique identifier for the memory
        value : Any
            Information to store
        category : str
            Memory category for organization
        importance : float
            Importance score (0-1)
        """
        if key in self.memories:
            # Update existing
            entry = self.memories[key]
            entry.value = value
            entry.updated_at = datetime.now()
            entry.importance = max(entry.importance, importance)
        else:
            # Create new
            entry = MemoryEntry(
                key=key,
                value=value,
                category=category,
                importance=importance
            )
            self.memories[key] = entry
            
            # Update category index
            if category not in self.categories:
                self.categories[category] = []
            self.categories[category].append(key)
    
    def retrieve(self, key: str) -> Optional[Any]:
        """
        Retrieve information from memory.
        
        Parameters
        ----------
        key : str
            Memory key
        
        Returns
        -------
        Optional[Any]
            Stored value or None
        """
        if key in self.memories:
            entry = self.memories[key]
            entry.access_count += 1
            return entry.value
        return None
    
    def get_by_category(self, category: str) -> Dict[str, Any]:
        """
        Get all memories in a category.
        
        Parameters
        ----------
        category : str
            Category name
        
        Returns
        -------
        Dict[str, Any]
            Key-value pairs for category
        """
        if category not in self.categories:
            return {}
        
        return {
            key: self.memories[key].value
            for key in self.categories[category]
            if key in self.memories
        }
    
    def get_important(self, threshold: float = 0.7, limit: int = 10) -> List[Tuple[str, Any]]:
        """
        Get most important memories.
        
        Parameters
        ----------
        threshold : float
            Minimum importance score
        limit : int
            Maximum number of results
        
        Returns
        -------
        List[Tuple[str, Any]]
            (key, value) pairs sorted by importance
        """
        important = [
            (entry.key, entry.value)
            for entry in self.memories.values()
            if entry.importance >= threshold
        ]
        important.sort(key=lambda x: self.memories[x[0]].importance, reverse=True)
        return important[:limit]
    
    def summarize(self) -> Dict[str, Any]:
        """Get memory statistics."""
        return {
            'total_memories': len(self.memories),
            'categories': {cat: len(keys) for cat, keys in self.categories.items()},
            'most_accessed': sorted(
                self.memories.items(),
                key=lambda x: x[1].access_count,
                reverse=True
            )[:5]
        }


# Test long-term memory
ltm = LongTermMemory()

# Store user preferences
ltm.store("user_name", "Alice", category="user_profile", importance=1.0)
ltm.store("preferred_language", "Python", category="user_profile", importance=0.9)
ltm.store("expertise_level", "intermediate", category="user_profile", importance=0.9)

# Store learned facts
ltm.store("project_name", "DataPipeline", category="context", importance=0.8)
ltm.store("api_key_location", "~/.config/app/key.txt", category="technical", importance=0.6)

print("Long-Term Memory Contents:")
print("=" * 60)

print("\nUser Profile:")
profile = ltm.get_by_category("user_profile")
for key, value in profile.items():
    print(f"  {key}: {value}")

print("\nImportant Memories:")
for key, value in ltm.get_important(threshold=0.8):
    print(f"  {key}: {value}")

print("\nMemory Summary:")
summary = ltm.summarize()
print(f"  Total memories: {summary['total_memories']}")
print(f"  Categories: {summary['categories']}")

## Part 3: Episodic Memory - Storing Experiences

Episodic memory stores specific experiences and events:

- **Conversations**: Full interaction histories
- **Actions**: Tasks completed and their outcomes
- **Errors**: Failed attempts and lessons learned

Uses embeddings for semantic search.

In [ ]:
# Purpose: Implement episodic memory with semantic search
# Key Concept: Retrieve relevant past experiences based on similarity

@dataclass
class Episode:
    """A single episodic memory."""
    id: str
    content: str
    embedding: List[float]
    metadata: Dict[str, Any]
    timestamp: datetime = field(default_factory=datetime.now)
    importance: float = 1.0


class EpisodicMemory:
    """
    Episodic memory with semantic search.
    """
    
    def __init__(self):
        """Initialize episodic memory."""
        self.episodes: List[Episode] = []
        self.next_id = 0
    
    def add_episode(
        self,
        content: str,
        metadata: Optional[Dict[str, Any]] = None,
        importance: float = 1.0
    ) -> str:
        """
        Add an episode to memory.
        
        Parameters
        ----------
        content : str
            Episode description
        metadata : Optional[Dict[str, Any]]
            Additional information
        importance : float
            Importance score
        
        Returns
        -------
        str
            Episode ID
        """
        # Generate embedding
        embedding = get_embedding(content)
        
        episode = Episode(
            id=f"ep_{self.next_id}",
            content=content,
            embedding=embedding,
            metadata=metadata or {},
            importance=importance
        )
        
        self.episodes.append(episode)
        self.next_id += 1
        
        return episode.id
    
    def search(
        self,
        query: str,
        top_k: int = 5,
        min_similarity: float = 0.7
    ) -> List[Tuple[Episode, float]]:
        """
        Search for relevant episodes.
        
        Parameters
        ----------
        query : str
            Search query
        top_k : int
            Number of results to return
        min_similarity : float
            Minimum similarity threshold
        
        Returns
        -------
        List[Tuple[Episode, float]]
            (episode, similarity_score) pairs
        """
        if not self.episodes:
            return []
        
        # Get query embedding
        query_embedding = get_embedding(query)
        
        # Calculate similarities
        results = []
        for episode in self.episodes:
            similarity = cosine_similarity(query_embedding, episode.embedding)
            if similarity >= min_similarity:
                results.append((episode, similarity))
        
        # Sort by similarity * importance
        results.sort(key=lambda x: x[1] * x[0].importance, reverse=True)
        
        return results[:top_k]
    
    def get_recent(self, n: int = 5) -> List[Episode]:
        """Get n most recent episodes."""
        return sorted(self.episodes, key=lambda e: e.timestamp, reverse=True)[:n]
    
    def prune_old(
        self,
        max_age_days: int = 30,
        min_importance: float = 0.5
    ) -> int:
        """
        Remove old, unimportant episodes.
        
        Parameters
        ----------
        max_age_days : int
            Maximum age in days
        min_importance : float
            Minimum importance to keep regardless of age
        
        Returns
        -------
        int
            Number of episodes removed
        """
        cutoff_date = datetime.now() - timedelta(days=max_age_days)
        
        original_count = len(self.episodes)
        self.episodes = [
            ep for ep in self.episodes
            if ep.timestamp >= cutoff_date or ep.importance >= min_importance
        ]
        
        return original_count - len(self.episodes)


# Test episodic memory
episodic = EpisodicMemory()

# Add episodes from past interactions
episodes_data = [
    ("User asked how to read CSV files in Python. Recommended pandas.read_csv() method.",
     {"topic": "file_io", "outcome": "success"}, 0.8),
    ("User had trouble with file encoding. Suggested using encoding='utf-8' parameter.",
     {"topic": "file_io", "outcome": "success"}, 0.9),
    ("Helped debug a sorting algorithm. Issue was incorrect comparison operator.",
     {"topic": "algorithms", "outcome": "success"}, 0.7),
    ("User asked about async/await in Python. Explained event loop and coroutines.",
     {"topic": "async", "outcome": "success"}, 0.6),
]

for content, metadata, importance in episodes_data:
    episodic.add_episode(content, metadata, importance)

print("Episodic Memory Search:")
print("=" * 60)

# Search for relevant past experiences
query = "How do I handle different file encodings when reading files?"
print(f"\nQuery: {query}")
print("\nRelevant past experiences:")

results = episodic.search(query, top_k=3, min_similarity=0.6)
for episode, similarity in results:
    print(f"\nSimilarity: {similarity:.3f}")
    print(f"Content: {episode.content}")
    print(f"Metadata: {episode.metadata}")

## Part 4: Memory Integration - Combining Memory Types

Production agents use all three memory types together.

In [ ]:
# Purpose: Integrate all memory types into unified system
# Key Concept: Different memory types serve different purposes

class UnifiedMemory:
    """
    Integrated memory system combining short-term, long-term, and episodic memory.
    """
    
    def __init__(self, system_prompt: str):
        """Initialize unified memory system."""
        self.short_term = ShortTermMemory(max_messages=10, system_prompt=system_prompt)
        self.long_term = LongTermMemory()
        self.episodic = EpisodicMemory()
    
    def process_interaction(
        self,
        user_input: str,
        include_relevant_episodes: bool = True
    ) -> str:
        """
        Process user input with full memory context.
        
        Parameters
        ----------
        user_input : str
            User's message
        include_relevant_episodes : bool
            Whether to search episodic memory
        
        Returns
        -------
        str
            Agent's response
        """
        # Build context from all memory types
        context_parts = []
        
        # Add relevant long-term memories
        user_profile = self.long_term.get_by_category("user_profile")
        if user_profile:
            context_parts.append("User Profile: " + json.dumps(user_profile))
        
        # Add relevant episodes
        if include_relevant_episodes:
            relevant_episodes = self.episodic.search(user_input, top_k=2, min_similarity=0.7)
            if relevant_episodes:
                context_parts.append("\nRelevant past experiences:")
                for ep, sim in relevant_episodes:
                    context_parts.append(f"- {ep.content}")
        
        # Add context to conversation
        if context_parts:
            context_message = "\n".join(context_parts)
            # Add as system message or prepend to user message
            enhanced_input = f"{context_message}\n\nCurrent query: {user_input}"
        else:
            enhanced_input = user_input
        
        # Add to short-term memory
        self.short_term.add_message("user", enhanced_input)
        
        # Get response
        response = call_llm(self.short_term.get_context())
        
        # Add response to short-term memory
        self.short_term.add_message("assistant", response)
        
        return response
    
    def save_interaction_as_episode(
        self,
        summary: str,
        metadata: Optional[Dict] = None,
        importance: float = 1.0
    ) -> None:
        """Save current interaction to episodic memory."""
        self.episodic.add_episode(summary, metadata, importance)
    
    def get_memory_summary(self) -> Dict[str, Any]:
        """Get summary of all memory systems."""
        return {
            'short_term': {
                'messages': len(self.short_term.messages),
                'max_capacity': self.short_term.max_messages
            },
            'long_term': self.long_term.summarize(),
            'episodic': {
                'total_episodes': len(self.episodic.episodes),
                'recent_count': len(self.episodic.get_recent(5))
            }
        }


# Test unified memory system
unified = UnifiedMemory("You are a helpful Python programming assistant.")

# Set up user profile
unified.long_term.store("user_name", "Bob", "user_profile", importance=1.0)
unified.long_term.store("skill_level", "intermediate", "user_profile", importance=0.9)

# Add some past episodes
unified.episodic.add_episode(
    "Helped user debug file reading issue with encoding problems",
    {"outcome": "success", "topic": "file_io"},
    importance=0.8
)

print("Testing Unified Memory System:")
print("=" * 60)

# Interact with agent
user_query = "I'm having trouble reading a file. Getting encoding errors."
print(f"\nUser: {user_query}")

response = unified.process_interaction(user_query)
print(f"\nAssistant: {response}")

# Save this interaction
unified.save_interaction_as_episode(
    "Helped user with file encoding issue, recommended utf-8 encoding",
    {"topic": "file_io", "outcome": "success"},
    importance=0.7
)

print("\n\nMemory Summary:")
print(json.dumps(unified.get_memory_summary(), indent=2))

## Exercises

### Exercise 3.1.1: Implement Memory Summarization

**Task:** Create a function that summarizes conversation history when short-term memory is full.

**Requirements:**
- Take list of messages
- Use LLM to generate concise summary
- Preserve key facts and decisions
- Return summary message

**Expected Output:** Summarized conversation context

In [ ]:
# YOUR CODE HERE
# ---------------

def summarize_conversation(messages: List[Message]) -> str:
    """
    Summarize conversation history.
    
    Parameters
    ----------
    messages : List[Message]
        Messages to summarize
    
    Returns
    -------
    str
        Summary of conversation
    """
    # Replace with your implementation
    return ""


# Test
# test_messages = memory.get_recent(10)
# summary = summarize_conversation(test_messages)
# print("Summary:", summary)

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_3_1_1():
    test_msgs = [
        Message("user", "Hello"),
        Message("assistant", "Hi there!"),
        Message("user", "How are you?"),
    ]
    result = summarize_conversation(test_msgs)
    assert isinstance(result, str), "Should return a string"
    assert len(result) > 0, "Summary should not be empty"
    assert len(result) < sum(len(m.content) for m in test_msgs), "Summary should be shorter than original"
    print("✅ Exercise 3.1.1 passed!")

test_exercise_3_1_1()

### Exercise 3.1.2: Build Memory Decay System

**Task:** Implement importance decay over time for episodic memories.

**Requirements:**
- Reduce importance based on age
- More important memories decay slower
- Frequently accessed memories decay slower
- Return updated importance score

**Expected Output:** Function that calculates decayed importance

In [ ]:
# YOUR CODE HERE
# ---------------

def calculate_memory_importance(
    initial_importance: float,
    age_days: float,
    access_count: int,
    decay_rate: float = 0.1
) -> float:
    """
    Calculate current importance with decay.
    
    Parameters
    ----------
    initial_importance : float
        Original importance (0-1)
    age_days : float
        Days since creation
    access_count : int
        Number of times accessed
    decay_rate : float
        Decay rate per day
    
    Returns
    -------
    float
        Current importance (0-1)
    """
    # Replace with your implementation
    return initial_importance


# Test
# importance = calculate_memory_importance(
#     initial_importance=0.9,
#     age_days=10,
#     access_count=5,
#     decay_rate=0.05
# )
# print(f"Current importance: {importance:.3f}")

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_3_1_2():
    # Test decay
    result = calculate_memory_importance(1.0, age_days=10, access_count=0)
    assert result < 1.0, "Importance should decay over time"
    assert result > 0.0, "Importance should not go negative"
    
    # Test access boost
    result_accessed = calculate_memory_importance(1.0, age_days=10, access_count=5)
    result_not_accessed = calculate_memory_importance(1.0, age_days=10, access_count=0)
    assert result_accessed >= result_not_accessed, "Accessed memories should decay slower"
    
    print("✅ Exercise 3.1.2 passed!")

test_exercise_3_1_2()

### Exercise 3.1.3: Create Memory Consolidation

**Task:** Build a function that consolidates similar episodic memories.

**Requirements:**
- Find similar episodes (high embedding similarity)
- Merge them into single consolidated memory
- Preserve important details from both
- Update importance based on frequency

**Expected Output:** Consolidated episode

In [ ]:
# YOUR CODE HERE
# ---------------

def consolidate_memories(
    memory_system: EpisodicMemory,
    similarity_threshold: float = 0.85
) -> int:
    """
    Consolidate similar episodic memories.
    
    Parameters
    ----------
    memory_system : EpisodicMemory
        Memory system to consolidate
    similarity_threshold : float
        Minimum similarity for consolidation
    
    Returns
    -------
    int
        Number of memories consolidated
    """
    # Replace with your implementation
    return 0


# Test
# test_episodic = EpisodicMemory()
# test_episodic.add_episode("User asked about file reading")
# test_episodic.add_episode("User asked how to read files")
# consolidated = consolidate_memories(test_episodic)
# print(f"Consolidated {consolidated} memories")

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_3_1_3():
    test_mem = EpisodicMemory()
    test_mem.add_episode("Help with Python")
    test_mem.add_episode("Help with Python programming")
    
    original_count = len(test_mem.episodes)
    consolidated = consolidate_memories(test_mem, similarity_threshold=0.7)
    
    assert isinstance(consolidated, int), "Should return number of consolidations"
    assert consolidated >= 0, "Consolidation count should be non-negative"
    
    print("✅ Exercise 3.1.3 passed!")

test_exercise_3_1_3()

### Exercise 3.1.4: Implement Memory Export/Import

**Task:** Create functions to persist memory to disk and reload it.

**Requirements:**
- Export all memory types to JSON format
- Handle embeddings (convert to list)
- Import and reconstruct memory objects
- Preserve all metadata and timestamps

**Expected Output:** Save/load functions for unified memory

In [ ]:
# YOUR CODE HERE
# ---------------

def export_memory(memory_system: UnifiedMemory) -> Dict[str, Any]:
    """
    Export unified memory to dictionary.
    
    Parameters
    ----------
    memory_system : UnifiedMemory
        Memory system to export
    
    Returns
    -------
    Dict[str, Any]
        Serializable memory data
    """
    # Replace with your implementation
    return {}


def import_memory(data: Dict[str, Any], system_prompt: str) -> UnifiedMemory:
    """
    Import unified memory from dictionary.
    
    Parameters
    ----------
    data : Dict[str, Any]
        Exported memory data
    system_prompt : str
        System prompt for short-term memory
    
    Returns
    -------
    UnifiedMemory
        Reconstructed memory system
    """
    # Replace with your implementation
    return UnifiedMemory(system_prompt)


# Test
# exported = export_memory(unified)
# print(f"Exported memory: {len(json.dumps(exported))} characters")
# reimported = import_memory(exported, "Test prompt")
# print(f"Reimported: {reimported.get_memory_summary()}")

In [ ]:
# AUTO-GRADED TESTS - Do not modify
# ----------------------------------
def test_exercise_3_1_4():
    test_unified = UnifiedMemory("Test")
    test_unified.long_term.store("test_key", "test_value", "test_category")
    
    exported = export_memory(test_unified)
    assert isinstance(exported, dict), "Export should return a dictionary"
    
    reimported = import_memory(exported, "Test")
    assert isinstance(reimported, UnifiedMemory), "Import should return UnifiedMemory"
    
    print("✅ Exercise 3.1.4 passed!")

test_exercise_3_1_4()

## Summary

**Key Takeaways:**

1. **Short-term memory** maintains conversation context with windowing to manage token limits
2. **Long-term memory** stores persistent facts organized by category and importance
3. **Episodic memory** records specific experiences with semantic search via embeddings
4. **Memory management** requires strategies for decay, consolidation, and pruning
5. **Integrated systems** combine all memory types for coherent, context-aware agents

**Best Practices:**
- Set appropriate window sizes based on task complexity and token budgets
- Use importance scores to prioritize what to remember
- Implement decay to prevent stale information
- Consolidate similar memories to reduce redundancy
- Persist critical memories to disk for session continuity

**What's Next:**
- Module 3.2: RAG Pipeline - Complete retrieval-augmented generation implementation
- Module 3.3: Advanced Retrieval - Hybrid search, reranking, query expansion

**Additional Resources:**
- [LangChain Memory](https://python.langchain.com/docs/modules/memory/)
- [OpenAI Embeddings Guide](https://platform.openai.com/docs/guides/embeddings)
- [MemGPT: Towards LLMs as Operating Systems](https://arxiv.org/abs/2310.08560)